<div align="right"><a href="S14-Quantization-of-filter-coefficients.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L14-Quantisierung-der-Filterkoeffizienten.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L13 Entwurf rekursiver Filter](./L13-Entwurf-rekursiver-Filter.ipynb) | [Aufgabe 14](./A14-Quantisierung-der-Filterkoeffizienten.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L15 Fensterfunktionen](./L15-Fensterfunktionen.ipynb)

</div>

---


# Quantisierung von Filterkoeffizienten

In Abschnitt 4.4.1 wurde während der Analyse der **Parameterquantisierung** gezeigt, dass die Pole empfindlicher werden, je mehr Pole vorhanden sind und je näher sie zueinander liegen. Bei den Nullstellen ist ein ähnliches Verhalten zu beobachten. Daher ist die **dritte kanonische Form** (Übung 4.4.1) von großer Bedeutung für die Realisierung höherer Ordnung von Systemen.

Diese Aufgabe zeigt ein Beispiel, wie die **Quantisierung von Filterkoeffizienten** die Systemfunktion $H(z)$ beeinflusst. Die Funktion `lfxquant()` implementiert einen linearen Quantisierer:

$$
-1 \leq [v]_Q \leq 1 - 2^{-l+1},
$$

wobei $l$ die Anzahl der Bits ist und $v$ die zu quantisierende Zahl darstellt. Um die Koeffizienten eines digitalen Filters zu quantisieren, müssen die Koeffizienten **normalisiert** werden. Die Funktion `[aq, n] = lcoefrnd(a, l)` berechnet den Vektor der quantisierten Koeffizienten `aq` sowie den Normalisierungsfaktor `n`.

---

### a)

Für unsere Analyse verwenden wir einen **Cauer-Tiefpassfilter** 7. Ordnung mit den folgenden Spezifikationen:

* $3,\text{dB}$-Eckfrequenz: $0.4 \pi$
* Durchlassbereichsdämpfung: $R_P = 0.1,\text{dB}$
* Sperrbereichsdämpfung: $R_S = 40,\text{dB}$

Der MATLAB-Befehl

```matlab
[b, a] = ellip(7, 0.1, 40, 0.4)
```

berechnet die Koeffizienten. Plotten Sie die **Frequenzgangfunktion** mit `freqz()`.

---

### b)

Quantisieren Sie die Filterkoeffizienten mit **8 Bits**, d.h.:

```matlab
[bq, nb] = lcoefrnd(b, 8)
[aq, na] = lcoefrnd(a, 8)
```

Dies ergibt die quantisierten Koeffizienten `bq` und `aq` zusammen mit den Normalisierungsfaktoren `nb` und `na`.

Plotten Sie die resultierenden Frequenzgänge sowie die Version mit nicht quantisierten Koeffizienten (Übung 4.4.1.a).

**Hinweis:** Um die Übertragungsfunktion des Filters mit quantisierten Parametern zu plotten, muss die Normalisierung der Koeffizienten invertiert werden:

```matlab
Hq = nb/na * freqz(bq, aq, 512)
```

---

### c)

Bei Anwendung der **8-Bit-Quantisierung** ergibt sich eine große Abweichung der Übertragungsfunktion im Vergleich zum vorgegebenen **Toleranzschema**. Die zulässigen Abweichungen sind:

* $R_D^+ = 0.15,\text{dB}$
* $R_D^- = -0.15,\text{dB}$
* $R_S = -38,\text{dB}$

> $R_D^+$ und $R_D^-$ bezeichnen die maximalen Schwankungen im Durchlassbereich, wobei $R_D^- = R_P$.

Bestimmen Sie die **minimale Wortlänge** der Filterkoeffizienten, um dieses Toleranzschema einzuhalten.

---

Es gibt verschiedene Strukturen digitaler Filter. Die **Direktform** wird in MATLAB mit der Funktion `filter()` implementiert, die zuvor verwendet wurde.

Um die Effekte der Quantisierung zu reduzieren, wird meist die **dritte kanonische Form** verwendet, bei der das System aus einer Reihe von erst- und zweitordentlichen Filtern in Serie realisiert wird.

Im Folgenden analysieren wir die Effekte der **Parameterquantisierung**, wenn die **dritte kanonische Form** für den oberen Cauer-Tiefpass verwendet wird.

---

### d)

Berechnen Sie die Filterkoeffizienten der **zweiten Ordnung Sektionen (SOS)** des Cauer-Tiefpasses mit `lcascade()` und die resultierende Systemfunktion:

$$
H(z) = \prod_{i=1}^{p} H_i(z)
$$

Quantisieren Sie die Filterkoeffizienten der SOS mit `lcoefrnd()` für Wortlängen $l = 8, 10, 12, \dots$ Bits und berechnen Sie die Übertragungsfunktion der Kaskade mit quantisierten Koeffizienten.

Bestimmen Sie, wie viele Bits notwendig sind, um die Filterkoeffizienten der SOS zu quantisieren und das **Toleranzschema** (Übung 4.4.1.a) einzuhalten. Vergleichen Sie diese Länge mit der minimalen Wortlänge, die für die **direkte Implementierung** des Filters nötig war, um das Toleranzschema zu erfüllen (Übung 4.4.1.c).

In [ ]:
# BEGIN SOLUTION
% ##  Benoetigte(s) m-File(s):  lfxquant.m, lcoefrnd.m, lcascade.m            ##

% #####  Teilaufgabe a:  Cauer Tiefpass  #####
[b a] = ellip(7, 0.1, 40, 0.4);
[H W] = freqz(b,a,512);

% # grafische Ausgabe # %
figure; 
plot(W/pi,20*log10(abs(H))); 
grid; 
xlabel('Omega/pi (Nyquist = 1)'); 
ylabel('|H(exp(j*Omega))| in dB'); 
title('Cauer-TP');

% #####  Teilaufgabe b:  Filterkoeffizienten quantisieren  #####
[b a] = ellip(7, 0.1, 40, 0.4);
[H W] = freqz(b,a,512);
[bq nb] = lcoefrnd(b,8);
[aq na] = lcoefrnd(a,8);
Hq = nb/na*freqz(bq,aq,512);

% # grafische Ausgabe # %
figure; 
plot(W/pi,20*log10(abs(H))); 
hold on; 
plot(W/pi,20*log10(abs(Hq)),'-.'); 
hold off; 
grid; 
title('- unquantisierte Koeff.,  -. 8 Bit quantisierte Koeff.'); 
xlabel('Omega/pi (Nyquist = 1)');
ylabel('|H(exp(j*Omega))| in dB');

% #####  Teilaufgabe c:  minimale Wortlaenge bestimmen  #####
[b a] = ellip(7,0.1,40,0.4);   % Filter entwerfen
[H W] = freqz(b,a,512);        % unquantisierte Uebertragungsfunktion
v_laenge = [8,12,16];

for l=v_laenge
    [bq,nb] = lcoefrnd(b,l);     % Koeffizienten mit l Bits quantisieren
    [aq,na] = lcoefrnd(a,l);
    Hq = nb/na*freqz(bq,aq,512); % quantisierte Uebertragungsfunktion
    
    % # grafische Ausgabe # %
    % Durchlassbereich
    figure;                      
    plot(W(1:208)/pi,20*log10(abs(H(1:208)))); 
    axis([0 0.5 -1.4 0.3]); 
    grid; 
    hold on;
    plot(W(1:208)/pi,-0.15*ones(size(Hq(1:208))),'w' ); %Toleranzgrenzen darstellen
    plot(W(1:208)/pi,+0.15*ones(size(Hq(1:208))),'w' );
    plot(W(1:208)/pi,20*log10(abs(Hq(1:208))),'--'); 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)');
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('Durch.: - unquantisiert,  -- quantisiert mit %d Bits', l);
    title(string);
    
    % Sperrbereich
    figure;                      
    plot(W(215:512)/pi,20*log10(abs(H(215:512)))); 
    hold on;
    plot(W(215:512)/pi,20*log10(abs(Hq(215:512))),'--');
    plot(W(215:512)/pi,-38*ones(size(Hq(215:512))),'w' ); 
    grid;
    hold off; 
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('Sperr.: - unquantisiert,  -- quantisiert mit %d Bits', l);
    title(string);
end

% #####  Teilaufgabe d:  Filterkoeffizienten quantisieren  #####
[b a] = ellip(7,0.1,40,0.4);  % Filter entwerfen
[B A] = lcascade(b,a);        % unquantisierte Uebertragungsfunktion in
                              % 3-ter kanonischer Form berechnen
[m n] = size(B);              % Produkt der Uebertragungsfunkt. der Teilsysteme
Hi = zeros(512,m);

for k=1:m;
    Hi(:,k) = freqz(B(k,:),A(k,:),512);
end
H3k = Hi(:,1) .* Hi(:,2) .* Hi(:,3) .* Hi(:,4);
Bq = zeros(size(B));          % Vektoren initialisieren
Aq = Bq;
nb = zeros(m,1);
na = nb;
v_laenge=[12];

for l=v_laenge

    for k = 1:m;         % Koeffizienten der Teilsysteme mit l Bits quantisieren
        [Bq(k,:),nb(k)] = lcoefrnd(B(k,:),l);
        [Aq(k,:),na(k)] = lcoefrnd(A(k,:),l);
    end
    Hq = zeros(512,m);          % quantisierte Gesamtuebertragungsfunktion

    for k=1:m
        Hq(:,k) = nb(k)/na(k)*freqz(Bq(k,:),Aq(k,:),512);
    end
    Hqg = Hq(:,1) .* Hq(:,2) .* Hq(:,3) .* Hq(:,4);

    % # grafische Ausgabe # %
    % Durchlassbereich
    figure;                     
    plot(W(1:208)/pi,20*log10(abs(H3k(1:208))));
    axis([0 0.5 -1.4 0.3]); 
    grid; 
    hold on;
    %Toleranzgrenzen darstellen
    plot(W(1:208)/pi,-0.15*ones(size(Hqg(1:208))),'w' ); 
    plot(W(1:208)/pi,+0.15*ones(size(Hqg(1:208))),'w' );
    plot(W(1:208)/pi,20*log10(abs(Hqg(1:208))),'--'); 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('3. Kanonische: Durch.: - unquantisiert,  -- quantisiert mit %d Bits', l);
    title(string);
    
    % Sperrbereich
    figure;                     
    plot(W(215:512)/pi,20*log10(abs(H3k(215:512)))); 
    hold on;
    plot(W(215:512)/pi,20*log10(abs(Hqg(215:512))),'--');
    plot(W(215:512)/pi,-38*ones(size(Hqg(215:512))),'w' ); 
    grid; 
    hold off;
    xlabel('Omega/pi (Nyquist = 1)'); 
    ylabel('|H(exp(j*Omega))| in dB');
    string = sprintf('3. Kanonische: Sperr.: - unquantisiert,  -- quantisiert mit %d Bits', l);
    title(string);
end
# END SOLUTION

## Lösung: Quantisierung von Filterkoeffizienten

### a)-b)

Die Abbildung zeigt den **Frequenzgang** des Cauer-Tiefpassfilters.

<img src="./pngs/ldsv4a04_ab.png" width="250">

Wenn die Koeffizienten beider Polynome (Zähler und Nenner) gemäß

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1}
$$

mit 8 Bits quantisiert werden, ergibt sich die dargestellte Amplitudencharakteristik. Ein Vergleich der beiden Funktionen zeigt eine deutliche Variation im Übergang vom **Durchlass-** zum **Sperrbereich**. Das vorgegebene Toleranzschema wird vom System mit quantisierten Koeffizienten **nicht erfüllt**.

---

### c)

Unter Verwendung der Toleranzabweichungen $R_D^+$ und $R_D^-$ aus der Aufgabe werden für die Realisierung in **Direktform** als Einzel-System mindestens **16-Bit-Worte** benötigt.

Die folgenden Abbildungen zeigen den Durchlass- und Sperrbereich der quantisierten Übertragungsfunktion im Vergleich zur nicht quantisierten Version:

<img src="./pngs/ldsv4a04_c1.png" width="250">
<img src="./pngs/ldsv4a04_c2.png" width="250">

Bei einer Wortlänge von 16 Bit gibt es eine **kleine Abweichung** vor der abfallenden Kante im Durchlassbereich, während im Sperrbereich keine Variation auftritt.

---

### d)

Wenn der Cauer-Tiefpass (Aufgabenteil a) in **dritter kanonischer Form** realisiert wird und die **minimale Wortlänge** für die Quantisierung bestimmt wird, um das Toleranzschema einzuhalten, zeigt sich, dass eine **12-Bit-Quantisierung** ausreichend ist.

Im Vergleich zur direkten Implementierung (Aufgabenteil b) werden somit **4 Bits weniger** benötigt. Die Abbildungen zeigen die Amplitudencharakteristiken im **Sperr-** und **Durchlassbereich**:

<img src="./pngs/ldsv4a04_d1.png" width="250">
<img src="./pngs/ldsv4a04_d2.png" width="250">